<a href="https://colab.research.google.com/github/hanauert/Hausarbeit-GenAI/blob/main/05_stance_detection_llama31.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Load gold annotated dataset**

In [3]:
import pandas as pd
from transformers import pipeline

In [4]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
subsample_gold = pd.read_csv('/content/drive/MyDrive/FKM/stance_detection/gold_annotated.csv')

In [6]:
# Mapping dictionary
label_map = {
    1.0: "positiv",
    2.0: "negativ",
    3.0: "neutral"
}

# Apply the mapping
subsample_gold['gold_standard'] = subsample_gold['gold_standard'].map(label_map)

In [7]:
subsample_gold['lead'] = subsample_gold['body_text'].str.split('\n\n|\n').str[0]

#**Load Llama3.1:8b**

In [8]:
!curl https://ollama.ai/install.sh | sh

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13281    0 13281    0     0  47513      0 --:--:-- --:--:-- --:--:-- 47432
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [9]:
import subprocess

# Start Ollama service in the background
ollama_process = subprocess.Popen(["ollama", "serve"])

# Optional: check if it's running
print("Ollama server started in the background.")

Ollama server started in the background.


In [10]:
!curl http://localhost:11434

Ollama is running

In [11]:
!pip install -q ollama openai
import pandas as pd
import ollama
from ollama import chat

In [12]:
ollama.pull("llama3.1:8b")

ProgressResponse(status='success', completed=None, total=None, digest=None)

##**Define Metrics**

In [13]:
from sklearn.metrics import cohen_kappa_score, f1_score, accuracy_score, precision_score, recall_score
import pandas as pd

def evaluate_model(true, pred):
    return {
        'Cohens Kappa': cohen_kappa_score(true, pred),
        'F1': f1_score(true, pred, average='weighted'),
        'Accuracy': accuracy_score(true, pred),
        'Precision': precision_score(true, pred, average='weighted'),
        'Recall': recall_score(true, pred, average='weighted')
    }

##**Annotate with Llama3.1:8b - Prompt 1**

In [14]:
from ollama import chat
from pydantic import BaseModel
from typing import List, Literal
import json


class MigrationAnnotation(BaseModel):
   annotation: Literal['positiv', 'negativ', 'neutral']

def classify_with_llama_31_p1(text):
    messages = [
        {"role": "system","content": "Bitte hilf mir, die Haltung des folgenden Absatzes zu Migration zu klassifizieren."},
        {"role": "user","content": f"{text} --- Wie ist die Haltung des Autors zu Migration?"}
    ]

    response = chat(
        messages=messages,
        model="llama3.1:8b",
        format=MigrationAnnotation.model_json_schema(),
        options=dict(seed=42, temperature=0.0)
    )

    return response['message']['content']


###**Annotate Title + Lead + Paragraph**

In [15]:
# Apply the classification function to the dataframe
subsample_gold['label_llama31_zs_p1'] = (subsample_gold['title'] + "\n\n" + subsample_gold['lead'] + "\n\n" + subsample_gold['paragraph']).apply(classify_with_llama_31_p1)

subsample_gold['label_llama31_zs_p1'] = subsample_gold['label_llama31_zs_p1'].str.extract(r'"annotation"\s*:\s*"(\w+)"')

In [16]:
from sklearn.metrics import classification_report
print(classification_report(subsample_gold['gold_standard'], subsample_gold['label_llama31_zs_p1']))

              precision    recall  f1-score   support

     negativ       0.62      0.24      0.35        62
     neutral       0.53      0.69      0.60       109
     positiv       0.57      0.61      0.59        79

    accuracy                           0.55       250
   macro avg       0.57      0.51      0.51       250
weighted avg       0.57      0.55      0.53       250



In [17]:
metrics_llama_p1 = evaluate_model(subsample_gold['gold_standard'], subsample_gold['label_llama31_zs_p1'])
print(metrics_llama_p1)

print("Evaluation Metrics: Llama3.1:8b - Prompt 1")
for metric, value in metrics_llama_p1.items():
    print(f"{metric}: {value:.3f}")

{'Cohens Kappa': np.float64(0.2801686462028895), 'F1': 0.5331798262791324, 'Accuracy': 0.552, 'Precision': 0.5658531187122736, 'Recall': 0.552}
Evaluation Metrics: Llama3.1:8b - Prompt 1
Cohens Kappa: 0.280
F1: 0.533
Accuracy: 0.552
Precision: 0.566
Recall: 0.552


##**Annotate with Llama3.1:8b - Prompt 2**

In [18]:
from ollama import chat
from pydantic import BaseModel
from typing import List, Literal
import json

class MigrationAnnotation(BaseModel):
   annotation: Literal['positiv', 'negativ', 'neutral']


def classify_with_llama_31_p2(text):
    messages = [
    {
        "role": "system",
        "content": (
            "Stance Detection: Klassifiziere den folgenden Absatz.\n"
            "positiv: Der Text zeigt eine befürwortende Haltung gegenüber Migration.\n"
            "negativ: Migration wird im Text eher kritisch betrachtet.\n"
            "neutral: Der Text drückt keine eindeutige Meinung zu Migration aus."
        )
    },
    {
        "role": "user",
        "content": f"{text} --- Wie ist die Haltung des Autors zu Migration? "
    }
]

    response = chat(
        messages=messages,
        model="llama3.1:8b",
        format=MigrationAnnotation.model_json_schema(),
        options=dict(seed=42, temperature=0.0)
    )

    return response['message']['content']


##**Annotate Title + Lead + Paragraph**

In [19]:
# Apply the classification function to the dataframe
subsample_gold['label_llama31_zs_p2'] = (subsample_gold['title'] + "\n\n" + subsample_gold['lead'] + "\n\n" + subsample_gold['paragraph']).apply(classify_with_llama_31_p2)

subsample_gold['label_llama31_zs_p2'] = subsample_gold['label_llama31_zs_p2'].str.extract(r'"annotation"\s*:\s*"(\w+)"')

In [20]:
from sklearn.metrics import classification_report
print(classification_report(subsample_gold['gold_standard'], subsample_gold['label_llama31_zs_p2']))

              precision    recall  f1-score   support

     negativ       0.64      0.11      0.19        62
     neutral       0.47      0.97      0.63       109
     positiv       0.92      0.15      0.26        79

    accuracy                           0.50       250
   macro avg       0.68      0.41      0.36       250
weighted avg       0.65      0.50      0.41       250



In [21]:
metrics_llama_p2 = evaluate_model(subsample_gold['gold_standard'], subsample_gold['label_llama31_zs_p2'])

print("Evaluation Metrics: Llama3.1:8b - Prompt 2")
for metric, value in metrics_llama_p2.items():
    print(f"{metric}: {value:.3f}")

Evaluation Metrics: Llama3.1:8b - Prompt 2
Cohens Kappa: 0.136
F1: 0.406
Accuracy: 0.500
Precision: 0.654
Recall: 0.500


##**Save Labels**

In [22]:
labels_df = pd.read_csv('/content/drive/MyDrive/FKM/stance_detection/classification_results/results_NLI_classification.csv')

In [23]:
labels_df['label_llama31_zs_p1'] = subsample_gold['label_llama31_zs_p1']

In [24]:
labels_df.to_csv('/content/drive/MyDrive/FKM/stance_detection/classification_results/results_NLI_classification.csv', index=False)

In [25]:
labels_df.head()

,date,title,paragraph,newspaper,gold_standard,label_ml_p6,score_ml_p6,label_mlb_p8,score_mlb_p8,label_ls_p6,score_ls_p6,label_llama31_zs_p1,label_gemma3_zs_p22
0,2025-01-02,Wir verschenken zu viel Zeit,Das gilt übrigens auch für die demokratischen ...,FR,positiv,positiv,0.440892,positiv,0.999996,positiv,0.987128,positiv,negativ
1,2024-12-26,Das Armutsrisiko in Deutschland ist rückläufig...,Trotz des gewachsenen Migrantenanteils in der ...,Welt,neutral,neutral,0.625038,neutral,0.988387,positiv,0.415422,positiv,neutral
2,2023-11-26,Arbeitgeber erwarten Wohlstandsverlust durch P...,Die Unionsfraktion hatte stattdessen vorgeschl...,Welt,negativ,negativ,0.933522,negativ,0.999986,negativ,0.999362,neutral,negativ
3,2024-11-25,Das Ende einer Erfolgsstory; Kanzler Scholz fe...,Hauptgründe für die Aufwärtsbewegung sind die ...,Welt,positiv,positiv,0.411763,positiv,0.905089,negativ,0.996768,neutral,negativ
4,2024-09-10,„Investitionen entscheiden über die Leistungsf...,Die Aufnahme einer Erwerbsarbeit ist die beste...,FR,positiv,negativ,0.922741,positiv,0.999997,positiv,0.995294,positiv,negativ


#**Evaluate Metrics**

#**Metrics**

In [26]:
metrics_gemma_p22 = evaluate_model(labels_df['gold_standard'], labels_df['label_gemma3_zs_p22'])

In [27]:
metrics_df = pd.DataFrame([metrics_gemma_p22, metrics_llama_p1],
                          index=['gemma3:4b', 'llama3.1:8b',])

print(metrics_df.round(3))

             Cohens Kappa     F1  Accuracy  Precision  Recall
gemma3:4b            0.31  0.536     0.532      0.602   0.532
llama3.1:8b          0.28  0.533     0.552      0.566   0.552
